### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    Tracking agent behavior with logging, analytics, and debugging.
    Transforming prompts, tool selection, and output formatting.
    Adding retries, fallbacks, and early termination logic.
    Applying rate limits, guardrails, and PII detection.


In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b")    

Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

    Long-running conversations that exceed context windows.
    Multi-turn dialogues with extensive history.
    Applications where preserving full conversation context matters.


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)


In [4]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [5]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a0333bdc-6787-4477-b575-cb7186384b62'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2?"\n2.  **Identify Core Question:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Accuracy:** The calculation is correct. No tricks or hidden meanings.\n6.  **Output Generation:** "2 + 2 equals 4." (or simply "4")\n\nI\'ll keep it direct and accurate.✅\n</think>\n\n2 + 2 equals **4**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 17, 'total_tokens': 161, 'completion_time': 0.274002238, 'completion_tokens_details': None, 'prompt_time': 0.000931719, 'prompt_tokens_details': None, 'queue_time': 0.04813442, 'total_time': 0.274933957}, 'model_name': 'qwen/qwen3